# AutoNavLog

**地上準備専用**。このNotebookの値は運航資料や完成帳票ではありません。警告と根拠を確認し、別添8-1へ手書きで清書してください。

1 Projectを複数Notebookから同時編集しないでください。

## 1. 開始・環境確認
## 2. Projectの作成・読込
## 3. コース取込・編集
## 4. 飛行計画入力
## 5. Forecast Run選択
## 6. 計算実行
## 7. 警告・未確定項目の解消
## 8. 清書ビュー
## 9. 保存・Snapshot作成

In [ ]:
#@title AutoNavLog配布版を検証して準備
from __future__ import annotations

import ctypes.util
import hashlib
import importlib.metadata
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

VERSION = "0.2.1"
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RELEASES_ROOT = Path("/content/drive/MyDrive/AutoNavLog/releases")
    if not RELEASES_ROOT.is_dir():
        raise FileNotFoundError(
            "AutoNavLog配布フォルダが見つかりません。"
            f"確認先: {RELEASES_ROOT}"
        )
    available = sorted(
        (
            path.name
            for path in RELEASES_ROOT.iterdir()
            if path.is_dir() and all(part.isdigit() for part in path.name.split("."))
        ),
        key=lambda value: tuple(int(part) for part in value.split(".")),
    )
    if available and available[-1] != VERSION:
        print(f"新版 {available[-1]} があります。このNotebookは {VERSION} を継続使用します。")
    RELEASE_ROOT = RELEASES_ROOT / VERSION
    manifest_path = RELEASE_ROOT / "release-manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"配布manifestが見つかりません: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for item in manifest["files"]:
        relative = Path(item["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError(f"配布manifestに不正なpathがあります: {item['path']}")
        path = RELEASE_ROOT / relative
        if not path.is_file():
            raise FileNotFoundError(f"配布ファイルが見つかりません: {item['path']}")
        if hashlib.sha256(path.read_bytes()).hexdigest() != item["sha256"]:
            raise RuntimeError(f"配布ファイルのSHA-256が一致しません: {item['path']}")
    if ctypes.util.find_library("eccodes") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "libeccodes0"], check=True)
    wheel_paths = [
        RELEASE_ROOT / "wheels" / "jma_msm_wind-0.2.1-py3-none-any.whl",
        RELEASE_ROOT / "wheels" / "autonavlog-0.2.1-py3-none-any.whl",
    ]
    if not all(path.is_file() for path in wheel_paths):
        raise FileNotFoundError(f"指定版wheelが揃っていません: {wheel_paths}")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--force-reinstall",
            *(str(path) for path in wheel_paths),
        ],
        check=True,
    )
    observed_versions = {
        "autonavlog": importlib.metadata.version("autonavlog"),
        "jma-msm-wind": importlib.metadata.version("jma-msm-wind"),
    }
    if observed_versions != {"autonavlog": VERSION, "jma-msm-wind": "0.2.1"}:
        raise RuntimeError(f"インストール版が配布指定と一致しません: {observed_versions}")
else:
    RELEASE_ROOT = Path.cwd()
print(f"AutoNavLog {VERSION} / release root: {RELEASE_ROOT}")

In [ ]:
#@title NAV LOG地上準備UIを表示
from autonavlog.application import CalculationService, ProjectService
from autonavlog.performance import PerformanceRepository
from autonavlog.presentation import AutoNavLogApp
from autonavlog.storage import (
    AirportRepository,
    LocalProjectRepository,
    ReferenceDataCatalogRepository,
    ReferenceDataError,
)
from autonavlog.storage.drive import GoogleDriveProjectRepository
from autonavlog.weather import FakeWeatherProvider
from autonavlog.weather.msm_adapter import MsmWeatherProvider

if IN_COLAB:
    app_data = RELEASE_ROOT / "data" / "autonavlog"
    repository = GoogleDriveProjectRepository("/content/drive/MyDrive")
    reference_root = Path("/content/drive/MyDrive/AutoNavLog/reference-data")
    weather = MsmWeatherProvider(
        "/content/drive/MyDrive/AutoNavLog/cache",
        RELEASE_ROOT / "data" / "msm" / "terrain.npz",
    )
else:
    app_data = RELEASE_ROOT / "data"
    repository = LocalProjectRepository(RELEASE_ROOT / ".notebook-data" / "AutoNavLog")
    reference_root = RELEASE_ROOT / ".notebook-data" / "AutoNavLog" / "reference-data"
    weather = FakeWeatherProvider()

reference_data = ReferenceDataCatalogRepository(
    reference_root,
    bundled_default=app_data / "reference" / "default",
)
try:
    airports = AirportRepository.from_reference_catalog(reference_data.open_active())
except ReferenceDataError as error:
    print(f"参照データを読み込めません。空港候補は空です。原因: {error}")
    airports = AirportRepository([])
performance = PerformanceRepository.from_directory_for_application(app_data / "performance")
calculation = CalculationService(airports, performance)
projects = ProjectService(repository)
app = AutoNavLogApp(projects, calculation, weather, reference_data)
app.render()